# 02 GRPO Train Colab
Robust training pipeline with GRPO first and safe fallback so artifacts are always produced.

In [ ]:
import os
REPO_URL='https://github.com/Chirag0096/ShiftLog-Gym.git'
REPO_DIR='ShiftLog-Gym'
if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}
!pip -q install -U pip setuptools wheel
!pip -q uninstall -y unsloth unsloth_zoo || true
!rm -rf unsloth_compiled_cache
!pip -q install -e . pandas matplotlib seaborn datasets accelerate peft bitsandbytes transformers trl huggingface_hub wandb llm-blender mergekit


In [ ]:
import os
from getpass import getpass
from train.colab_training_pipeline import ColabTrainingPipeline

wandb_key=os.environ.get('WANDB_API_KEY','').strip() or getpass('Enter WANDB_API_KEY (blank to disable W&B): ').strip()
hf_token=os.environ.get('HF_TOKEN','').strip() or getpass('Enter HF_TOKEN (blank to skip HF login): ').strip()

pipe=ColabTrainingPipeline()
pipe.authenticate(wandb_key=wandb_key, hf_token=hf_token)
pipe.assert_clean_trl_runtime()
pipe.load_model(os.environ.get('SHIFTLOG_MODEL','Qwen/Qwen2.5-1.5B-Instruct'))
print('Pipeline initialized')


In [ ]:
RUN_STAGE_A=True
RUN_STAGE_B=True
RUN_STAGE_C=True
if RUN_STAGE_A:
    pipe.run_stage_a(enabled=True)
if RUN_STAGE_B:
    print(pipe.run_stage_grpo(pipe.stage_b))
if RUN_STAGE_C:
    print(pipe.run_stage_grpo(pipe.stage_c))


In [ ]:
summaries=pipe.evaluate_and_write()
print('Eval summaries:', summaries)
print('Artifact status:')
for path,status in pipe.artifact_status():
    print(path, '->', status)


In [ ]:
PUBLISH_TO_HF=False
HF_MODEL_REPO=os.environ.get('HF_MODEL_REPO','Chirag0096/shiftlog-gym-qwen2.5-1.5b-memory-policy')
if PUBLISH_TO_HF:
    pipe.upload_to_hf(HF_MODEL_REPO)
    print('Uploaded to', HF_MODEL_REPO)
else:
    print('Set PUBLISH_TO_HF=True to upload adapter + curves.')
